## dataloader

In [ ]:
# ── Method 2: Using the convenience function ────────────────────────────────
import numpy as np

splits_alt, features = load_dataset()

print("\n" + "=" * 60)
print("ALTERNATIVE: Using load_dataset() convenience function")
print("=" * 60)
print(f"Features: {features}")
print(f"Train rows: {len(splits_alt['train'])}")
print(f"Val rows: {len(splits_alt['val'])}")
print(f"Test rows: {len(splits_alt['test'])}")

# ── Method 3: Get all splits as numpy arrays at once ────────────────────────
all_splits_arrays = loader.get_all_splits(return_arrays=True)

print("\n" + "=" * 60)
print("NUMPY ARRAYS: Ready for model training")
print("=" * 60)
for split_name, (X, y) in all_splits_arrays.items():
    print(f"{split_name}: X {X.shape} (float32), y {y.shape} (int64)")
    print(f"  -> y classes: {np.unique(y)}, class ratio: {np.bincount(y)}")

## Bayesian Hyperparameter Optimization

In [ ]:
# Run XGBoost Bayesian Hyperparameter Optimization
# Using fixed split mode with 50 trials (production would use 100+)
print("Starting XGBoost Bayesian Hyperparameter Optimization...")
print("This may take 5-10 minutes depending on system")
print("=" * 60)

result_xgb = tune_xgboost(loader, n_trials=50, loso=False, verbose=True)
best_xgb_params = result_xgb["best_params"]
best_xgb_value = result_xgb["best_value"]

print("\n" + "=" * 60)
print("XGBoost Optimization Complete!")
print(f"Best F1 Score: {best_xgb_value:.4f}")
print("\nBest Parameters:")
for key, value in best_xgb_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Save best parameters to a documentation file
import json
from datetime import datetime

# Prepare results dictionary
hpo_results = {
    "timestamp": datetime.now().isoformat(),
    "xgboost": {"best_f1_score": float(best_xgb_value), "parameters": best_xgb_params},
    "catboost": {"best_f1_score": float(best_cat_value), "parameters": best_cat_params},
}

# Save to JSON file
results_file = "../outputs/hpo_best_parameters.json"
with open(results_file, "w") as f:
    json.dump(hpo_results, f, indent=2)

print(f"✓ Best parameters saved to: {results_file}")
print("\nSummary:")
print(f"  XGBoost F1: {best_xgb_value:.4f}")
print(f"  CatBoost F1: {best_cat_value:.4f}")
print(f"  Best Model: {'XGBoost' if best_xgb_value >= best_cat_value else 'CatBoost'}")
print(f"              (F1: {max(best_xgb_value, best_cat_value):.4f})")

In [ ]:
# Train and evaluate XGBoost with best hyperparameters
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

print("Training XGBoost with best hyperparameters...")
print("=" * 60)

# Get data
splits = splits_alt  # Using normalized data
X_train = splits["train"][loader.feature_cols].values
y_train = splits["train"][loader.target_col].values
X_val = splits["val"][loader.feature_cols].values
y_val = splits["val"][loader.target_col].values
X_test = splits["test"][loader.feature_cols].values
y_test = splits["test"][loader.target_col].values

# Build XGBoost model with best parameters
scale_pos_weight = (y_train == 0).sum() / ((y_train == 1).sum() + 1e-6)
xgb_model_best = xgb.XGBClassifier(
    **best_xgb_params, scale_pos_weight=scale_pos_weight, random_state=42, verbosity=0
)

# Train
xgb_model_best.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# Predict
y_pred_xgb = xgb_model_best.predict(X_test)
y_proba_xgb = xgb_model_best.predict_proba(X_test)[:, 1]

# Evaluate
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_proba_xgb)
xgb_prauc = average_precision_score(y_test, y_proba_xgb)

print("\nXGBoost Test Metrics:")
print(f"  F1 Score:     {xgb_f1:.4f}")
print(f"  Precision:    {xgb_precision:.4f}")
print(f"  Recall:       {xgb_recall:.4f}")
print(f"  ROC-AUC:      {xgb_auc:.4f}")
print(f"  PR-AUC:       {xgb_prauc:.4f}")

In [ ]:
# Compare results and save comparison
import pandas as pd

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING RESULTS COMPARISON")
print("=" * 60)

# Create comparison dataframe
comparison_df = pd.DataFrame(
    {
        "XGBoost": [xgb_f1, xgb_precision, xgb_recall, xgb_auc, xgb_prauc],
        "CatBoost": [cat_f1, cat_precision, cat_recall, cat_auc, cat_prauc],
    },
    index=["F1 Score", "Precision", "Recall", "ROC-AUC", "PR-AUC"],
)

print("\nTest Set Performance:")
print(comparison_df.round(4))

# Determine winner
winner = "XGBoost" if xgb_f1 >= cat_f1 else "CatBoost"
print(f"\n🏆 Best Model: {winner} (F1: {max(xgb_f1, cat_f1):.4f})")

# Save comparison to file
comparison_file = "../outputs/hpo_results_comparison.csv"
comparison_df.to_csv(comparison_file)
print(f"✓ Comparison saved to: {comparison_file}")

# Create detailed report
report = {
    "timestamp": datetime.now().isoformat(),
    "hpo_trials": 50,
    "hpo_mode": "fixed_split",
    "xgboost": {
        "hpo_best_f1": float(best_xgb_value),
        "test_metrics": {
            "f1_score": float(xgb_f1),
            "precision": float(xgb_precision),
            "recall": float(xgb_recall),
            "roc_auc": float(xgb_auc),
            "pr_auc": float(xgb_prauc),
        },
        "best_parameters": best_xgb_params,
    },
    "catboost": {
        "hpo_best_f1": float(best_cat_value),
        "test_metrics": {
            "f1_score": float(cat_f1),
            "precision": float(cat_precision),
            "recall": float(cat_recall),
            "roc_auc": float(cat_auc),
            "pr_auc": float(cat_prauc),
        },
        "best_parameters": best_cat_params,
    },
    "winner": winner,
}

# Save detailed report
report_file = "../outputs/hpo_detailed_report.json"
with open(report_file, "w") as f:
    json.dump(report, f, indent=2)

print(f"✓ Detailed report saved to: {report_file}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt  # noqa: F401  # prime inline backend before train_classifiers
import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import precision_recall_curve, roc_curve

from train_classifiers import (
    MODEL_NAMES,
    PLOTS_DIR,
    _format_comparison_fixed,
    _run_loso,
    _train_and_eval_fixed,
    plot_feature_importance,
    plot_pr_curves,
    plot_roc_curves,
)

In [ ]:
# Reuse the `loader` already prepared in the dataloader cells above
fixed_results = _train_and_eval_fixed(loader)
print(_format_comparison_fixed(fixed_results))

In [ ]:
# Feature importance for both fitted models
fitted_models_fixed = {name: fixed_results[name]["model"] for name in MODEL_NAMES}
plot_feature_importance(fitted_models_fixed, loader.feature_cols)
display(Image(str(PLOTS_DIR / "fig_feature_importance.png")))

In [ ]:
# `loader.df` is available after prepare() — no extra load_data() call needed
loso_results = _run_loso(loader)

In [ ]:
# LOSO pooled ROC + PR curves
roc_data_loso = {}
pr_data_loso = {}
for name in MODEL_NAMES:
    r = loso_results[name]
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_proba"])
    prec, rec, _ = precision_recall_curve(r["y_true"], r["y_proba"])
    roc_data_loso[name] = (fpr, tpr, r["overall"]["roc_auc"])
    pr_data_loso[name] = (prec, rec, r["overall"]["pr_auc"])

plot_roc_curves(roc_data_loso, suffix="_loso")
plot_pr_curves(pr_data_loso, suffix="_loso")
display(Image(str(PLOTS_DIR / "fig_roc_curves_loso.png")))
display(Image(str(PLOTS_DIR / "fig_pr_curves_loso.png")))

In [ ]:
# Identify the better-performing model on each evaluation protocol
print("=== Fixed Split — Test Set ===")
best_fixed = max(MODEL_NAMES, key=lambda n: fixed_results[n]["test"]["f1"])
for name in MODEL_NAMES:
    m = fixed_results[name]["test"]
    print(
        f"  {name:10s}: Prec={m['precision']:.4f}  Rec={m['recall']:.4f}  "
        f"F1={m['f1']:.4f}  ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}"
    )
print(f"\n  Best (F1): {best_fixed}")

print("\n=== LOSO-CV — Pooled (optimal threshold) ===")
best_loso = max(MODEL_NAMES, key=lambda n: loso_results[n]["overall"]["f1"])
for name in MODEL_NAMES:
    m = loso_results[name]["overall"]
    thr = loso_results[name]["opt_threshold"]
    print(
        f"  {name:10s}: Prec={m['precision']:.4f}  Rec={m['recall']:.4f}  "
        f"F1={m['f1']:.4f}  ROC-AUC={m['roc_auc']:.4f}  PR-AUC={m['pr_auc']:.4f}  "
        f"threshold={thr:.4f}"
    )
print(f"\n  Best (F1): {best_loso}")

## Threshold Adjustment to Minimize False Positive Rate

In [ ]:
import sys

sys.path.insert(0, "../src")
from sklearn.metrics import confusion_matrix

from threshold_utils import find_threshold_for_fpr

target_fpr = 0.05  # Allow max 5% false positive rate

print(f"Testing threshold adjustment for target FPR = {target_fpr:.2f}\n")

for model_name, model in [("XGBoost", xgb_model_best), ("CatBoost", cat_model_best)]:
    print(f"--- {model_name} ---")
    y_proba_test = model.predict_proba(X_test)[:, 1]

    new_threshold = find_threshold_for_fpr(y_test, y_proba_test, target_fpr)
    y_pred_new = (y_proba_test >= new_threshold).astype(int)

    cm = confusion_matrix(y_test, y_pred_new)
    print(f"New Threshold: {new_threshold:.4f}")
    print(f"Confusion Matrix:\n{cm}\n")